### 1.寻找最优参数 

#### 1.1 寻找最优分层,或则动态调整分层

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
import json
import math
import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# 路径配置 (根据你的环境修改)
project_root = "/home/wangshuo/projects/Neo4j_Exp"
if project_root not in sys.path:
    sys.path.append(project_root)

# 引入核心类
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler

def _worker_tune_k(
    agg_file: str,
    aggregated_dir: str,
    all_t_true: dict,
    k_candidates: list,
    run_times: int,
    fixed_budget_frac: float,
    config: dict
):
    """
    [子进程] 对单个查询文件，遍历所有 K 值进行测试
    """
    # 1. 解析查询名
    if agg_file.startswith("aggregated_list_"):
        base = agg_file.replace("aggregated_list_", "")
    elif agg_file.startswith("aggregated_wide_"):
        base = agg_file.replace("aggregated_wide_", "")
    else:
        base = agg_file
    query_basename = base.replace(".csv", "") + ".graph"

    T_true = all_t_true.get(query_basename)
    if T_true is None:
        return []

    filepath = os.path.join(aggregated_dir, agg_file)
    
    # 2. 初始化 Sampler (先用默认 K=1 初始化)
    try:
        sampler = ProxyStratifiedSampler(
            csv_path=filepath,
            is_multi_predicate=True,
            post_proxy=config["POST_PROXY"],
            comment_proxy=config["COMMENT_PROXY"],
            post_oracle=config["POST_ORACLE"],
            comment_oracle=config["COMMENT_ORACLE"],
            T_true=T_true,
            total_budget_frac=fixed_budget_frac, # 固定 0.1
            K=1,
            c_stage=0.0 # 关键：禁用 Pilot，保证 100% 预算用于分层
        )
    except Exception:
        return []

    if sampler.posts.empty:
        return []

    results = []

    # 3. 遍历 K 值
    for k_val in k_candidates:
        # 【关键】动态修改 K
        sampler.K = k_val
        
        # 运行多次取平均
        q_errors = []
        for _ in range(run_times):
            try:
                # 调用你的目标方法: 7_ProxyE_Imp_Sqrt_WP_NRS
                res = sampler.run_proxyE_alloc_sqrt_wp_nrs()
                q_errors.append(res["Qerror"])
            except Exception:
                pass
        
        if q_errors:
            mean_q = np.mean(q_errors)
            results.append({
                "query_basename": query_basename,
                "K": k_val,
                "Mean_Qerror": mean_q
            })
            
    return results

def run_k_tuning(
    dataset_name: str = "dataset_test3",
    run_times: int = 5,
    max_workers: int = None
):
    print(f"\n{'='*10} Hyperparameter Tuning: K (Strata Count) {'='*10}")
    print(f"Dataset: {dataset_name}")
    print(f"Method: 7_ProxyE_Imp_Sqrt_WP_NRS (c_stage=0.0)")
    
    # === 参数配置 ===
    FIXED_BUDGET = 0.1
    K_CANDIDATES = [1, 2, 5, 10, 15, 20, 30]
    
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    aggregated_dir = os.path.join(base_path, "results", "aggregated_results")
    t_true_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability.json")
    
    # 输出文件
    output_dir = os.path.join(base_path, "results", "efficiency")
    os.makedirs(output_dir, exist_ok=True)
    output_csv = os.path.join(output_dir, "tuning_k_results.csv")
    
    config = {
        "POST_PROXY": "ML1_proxy4b_probability",
        "COMMENT_PROXY": "ML2_proxy1_probability",
        "POST_ORACLE": "ML1_oracle2_probability",
        "COMMENT_ORACLE": "ML2_oracle2_probability"
    }

    # 加载 T_true
    if not os.path.exists(t_true_path):
        print("T_true not found.")
        return
    with open(t_true_path, 'r') as f:
        all_t_true = json.load(f)

    # 准备文件
    agg_files = sorted([f for f in os.listdir(aggregated_dir) if f.endswith(".csv")])
    
    # 初始化结果文件
    pd.DataFrame(columns=["query_basename", "K", "Mean_Qerror"]).to_csv(output_csv, index=False)

    # 并行执行
    if max_workers is None:
        max_workers = max(1, os.cpu_count() - 2)

    print(f"Searching K in: {K_CANDIDATES}")
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for agg_file in agg_files:
            futures.append(
                executor.submit(
                    _worker_tune_k,
                    agg_file, aggregated_dir, all_t_true, 
                    K_CANDIDATES, run_times, FIXED_BUDGET, config
                )
            )
        
        all_results = []
        for future in tqdm(as_completed(futures), total=len(futures), desc="Tuning"):
            res = future.result()
            if res:
                all_results.extend(res)
                # 实时写入
                pd.DataFrame(res).to_csv(output_csv, mode='a', header=False, index=False)

    # === 分析结果并打印 ===
    print("\n>>> Tuning Analysis <<<")
    df = pd.DataFrame(all_results)
    if df.empty:
        print("No results.")
        return

    # 计算每个 K 值下，所有查询误差的平均值 (Macro Average)
    # 以及 P90 (90分位误差，衡量稳健性)
    summary = df.groupby("K")["Mean_Qerror"].agg(
        Avg_Error='mean',
        P50_Error='median',
        P90_Error=lambda x: x.quantile(0.95)
    ).reset_index()
    
    print(summary.to_string(index=False))
    
    # 找到最佳 K (基于 Avg Error)
    best_k = summary.loc[summary["Avg_Error"].idxmin()]["K"]
    print(f"\n✅ Recommended K (lowest avg error): {int(best_k)}")
    
    # 找到最佳 K (基于 P90 Error - 更稳健)
    best_k_p90 = summary.loc[summary["P90_Error"].idxmin()]["K"]
    print(f"✅ Recommended K (lowest P90 error): {int(best_k_p90)}")

if __name__ == "__main__":
    run_k_tuning(dataset_name="dataset_test", run_times=5)


========== Hyperparameter Tuning: K (Strata Count) ==========
Dataset: dataset_test
Method: 7_ProxyE_Imp_Sqrt_WP_NRS (c_stage=0.0)
Searching K in: [1, 2, 5, 10, 15, 20, 30]


Tuning:   0%|          | 0/153 [00:00<?, ?it/s]

[Auto-Tune] Budget=72, Reduced K from 10 to 7
[Auto-Tune] Budget=72, Reduced K from 10 to 7
[Auto-Tune] Budget=72, Reduced K from 10 to 7
[Auto-Tune] Budget=72, Reduced K from 10 to 7
[Auto-Tune] Budget=72, Reduced K from 10 to 7
[Auto-Tune] Budget=72, Reduced K from 15 to 7
[Auto-Tune] Budget=72, Reduced K from 15 to 7
[Auto-Tune] Budget=72, Reduced K from 15 to 7
[Auto-Tune] Budget=72, Reduced K from 15 to 7
[Auto-Tune] Budget=104, Reduced K from 15 to 10[Auto-Tune] Budget=72, Reduced K from 15 to 7

[Auto-Tune] Budget=121, Reduced K from 15 to 12
[Auto-Tune] Budget=72, Reduced K from 20 to 7
[Auto-Tune] Budget=104, Reduced K from 15 to 10
[Auto-Tune] Budget=121, Reduced K from 15 to 12
[Auto-Tune] Budget=72, Reduced K from 20 to 7
[Auto-Tune] Budget=142, Reduced K from 15 to 14
[Auto-Tune] Budget=104, Reduced K from 15 to 10
[Auto-Tune] Budget=72, Reduced K from 20 to 7
[Auto-Tune] Budget=121, Reduced K from 15 to 12
[Auto-Tune] Budget=104, Reduced K from 15 to 10
[Auto-Tune] Budget

Tuning:   7%|▋         | 10/153 [00:02<00:31,  4.55it/s]

[Auto-Tune] Budget=104, Reduced K from 20 to 10
[Auto-Tune] Budget=190, Reduced K from 20 to 19
[Auto-Tune] Budget=142, Reduced K from 20 to 14
[Auto-Tune] Budget=153, Reduced K from 20 to 15
[Auto-Tune] Budget=121, Reduced K from 20 to 12
[Auto-Tune] Budget=104, Reduced K from 30 to 10
[Auto-Tune] Budget=183, Reduced K from 20 to 18
[Auto-Tune] Budget=142, Reduced K from 20 to 14
[Auto-Tune] Budget=104, Reduced K from 30 to 10
[Auto-Tune] Budget=190, Reduced K from 20 to 19[Auto-Tune] Budget=153, Reduced K from 20 to 15

[Auto-Tune] Budget=121, Reduced K from 20 to 12
[Auto-Tune] Budget=183, Reduced K from 20 to 18
[Auto-Tune] Budget=104, Reduced K from 30 to 10
[Auto-Tune] Budget=142, Reduced K from 20 to 14
[Auto-Tune] Budget=121, Reduced K from 30 to 12
[Auto-Tune] Budget=153, Reduced K from 20 to 15
[Auto-Tune] Budget=190, Reduced K from 20 to 19
[Auto-Tune] Budget=104, Reduced K from 30 to 10
[Auto-Tune] Budget=183, Reduced K from 20 to 18
[Auto-Tune] Budget=121, Reduced K from 3

...

[Auto-Tune] Budget=184, Reduced K from 30 to 18
[Auto-Tune] Budget=184, Reduced K from 30 to 18


Tuning:  90%|████████▉ | 137/153 [00:10<00:01, 14.77it/s]

[Auto-Tune] Budget=184, Reduced K from 30 to 18


Tuning: 100%|██████████| 153/153 [00:13<00:00, 11.00it/s]



>>> Tuning Analysis <<<
 K  Avg_Error  P50_Error  P90_Error
 1   0.109021   0.064426   0.283342
 2   0.118024   0.075953   0.318238
 5   0.114709   0.070824   0.283342
10   0.111926   0.068085   0.283342
15   0.112746   0.065088   0.283342
20   0.120621   0.075296   0.283342
30   0.117019   0.070844   0.283342

✅ Recommended K (lowest avg error): 1
✅ Recommended K (lowest P90 error): 1


In [ ]:
import os
import json
import math
import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# 路径配置 (根据你的环境修改)
project_root = "/home/wangshuo/projects/Neo4j_Exp"
if project_root not in sys.path:
    sys.path.append(project_root)

# 引入核心类
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler

def _worker_tune_k(
    agg_file: str,
    aggregated_dir: str,
    all_t_true: dict,
    k_candidates: list,
    run_times: int,
    fixed_budget_frac: float,
    config: dict
):
    """
    [子进程] 对单个查询文件，遍历所有 K 值，并测试两种分层方法
    """
    # 1. 解析查询名
    if agg_file.startswith("aggregated_list_"):
        base = agg_file.replace("aggregated_list_", "")
    elif agg_file.startswith("aggregated_wide_"):
        base = agg_file.replace("aggregated_wide_", "")
    else:
        base = agg_file
    query_basename = base.replace(".csv", "") + ".graph"

    T_true = all_t_true.get(query_basename)
    if T_true is None:
        return []

    filepath = os.path.join(aggregated_dir, agg_file)
    
    # 2. 初始化 Sampler (先用默认 K=1 初始化)
    try:
        sampler = ProxyStratifiedSampler(
            csv_path=filepath,
            is_multi_predicate=True,
            post_proxy=config["POST_PROXY"],
            comment_proxy=config["COMMENT_PROXY"],
            post_oracle=config["POST_ORACLE"],
            comment_oracle=config["COMMENT_ORACLE"],
            T_true=T_true,
            total_budget_frac=fixed_budget_frac, # 固定预算
            K=1,
            c_stage=0.0 # 关键：禁用 Pilot
        )
    except Exception:
        return []

    if sampler.posts.empty:
        return []

    results = []

    # === 定义要对比的方法 ===
    # 确保你的 ProxyStratifiedSampler 类中已经实现了 run_proxyE_cluster_sqrt_wp_nrs
    methods_to_tune = {
        # "EqualDepth_NRS": sampler.run_proxyE_alloc_sqrt_wp_nrs,   # 等频分层
        "Cluster_NRS": sampler.run_proxyE_cluster_sqrt_wp_nrs     # 聚类分层
    }

    # 3. 遍历 K 值
    for k_val in k_candidates:
        # 【关键】动态修改 K
        sampler.K = k_val
        
        # 4. 遍历两种方法
        for method_name, method_func in methods_to_tune.items():
            q_errors = []
            for _ in range(run_times):
                try:
                    res = method_func()
                    q_errors.append(res["Qerror"])
                except Exception:
                    pass
            
            if q_errors:
                mean_q = np.mean(q_errors)
                results.append({
                    "query_basename": query_basename,
                    "Method": method_name,  # 新增列：方法名
                    "K": k_val,
                    "Mean_Qerror": mean_q
                })
            
    return results

def run_k_tuning(
    dataset_name: str = "dataset_three",
    run_times: int = 5,
    max_workers: int = None
):
    print(f"\n{'='*10} Hyperparameter Tuning: K (EqualDepth vs Cluster) {'='*10}")
    print(f"Dataset: {dataset_name}")
    
    # === 参数配置 ===
    FIXED_BUDGET = 0.1
    K_CANDIDATES = [1,2,5,10,15,20,25,30]
    
    base_path = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
    aggregated_dir = os.path.join(base_path, "results", "aggregated_results")
    t_true_path = os.path.join(base_path, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability.json")
    
    # 输出文件
    output_dir = os.path.join(base_path, "results", "efficiency")
    os.makedirs(output_dir, exist_ok=True)
    output_csv = os.path.join(output_dir, "tuning_k_comparison_results.csv")
    
    config = {
        "POST_PROXY": "ML1_proxy4b_probability",
        "COMMENT_PROXY": "ML2_proxy1_probability",
        "POST_ORACLE": "ML1_oracle2_probability",
        "COMMENT_ORACLE": "ML2_oracle2_probability"
    }

    # 加载 T_true
    if not os.path.exists(t_true_path):
        print("T_true not found.")
        return
    with open(t_true_path, 'r') as f:
        all_t_true = json.load(f)

    # 准备文件
    if not os.path.exists(aggregated_dir):
        print("Aggregated dir not found.")
        return
    agg_files = sorted([f for f in os.listdir(aggregated_dir) if f.endswith(".csv")])
    
    # 初始化结果文件 (增加 Method 列)
    pd.DataFrame(columns=["query_basename", "Method", "K", "Mean_Qerror"]).to_csv(output_csv, index=False)

    # 并行执行
    if max_workers is None:
        max_workers = max(1, os.cpu_count() - 2)

    print(f"Searching K in: {K_CANDIDATES}")
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for agg_file in agg_files:
            futures.append(
                executor.submit(
                    _worker_tune_k,
                    agg_file, aggregated_dir, all_t_true, 
                    K_CANDIDATES, run_times, FIXED_BUDGET, config
                )
            )
        
        all_results = []
        for future in tqdm(as_completed(futures), total=len(futures), desc="Tuning"):
            res = future.result()
            if res:
                all_results.extend(res)
                # 实时写入
                pd.DataFrame(res).to_csv(output_csv, mode='a', header=False, index=False)

    # === 分析结果并打印 ===
    print("\n>>> Tuning Analysis <<<")
    df = pd.DataFrame(all_results)
    if df.empty:
        print("No results.")
        return

    # 按方法分别统计
    unique_methods = df["Method"].unique()
    
    for method in unique_methods:
        print(f"\n--- Method: {method} ---")
        subset = df[df["Method"] == method]
        
        # 计算统计量
        summary = subset.groupby("K")["Mean_Qerror"].agg(
            Avg_Error='mean',
            P50_Error='median',
            P90_Error=lambda x: x.quantile(0.95)
        ).reset_index()
        
        print(summary.to_string(index=False))
        
        # 寻找最优 K
        best_k_avg = summary.loc[summary["Avg_Error"].idxmin()]["K"]
        best_k_p90 = summary.loc[summary["P90_Error"].idxmin()]["K"]
        
        print(f"✅ Best K (Avg): {int(best_k_avg)}")
        print(f"✅ Best K (P95): {int(best_k_p90)}")

if __name__ == "__main__":
    run_k_tuning(dataset_name="dataset_test", run_times=3)